In [19]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Resizing, Rescaling, Dense, BatchNormalization, Input, InputLayer, MaxPool2D, Flatten, Conv2D, Dropout
from tensorflow.keras.regularizers import L2, L1
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.metrics import TopKCategoricalAccuracy, CategoricalAccuracy
from tensorflow.keras.optimizers import Adam

In [1]:
Train_Directory = r"C:\Users\ABHAY SINHA\Parking_Lot_Project\clf-data\clf-data"

In [37]:
Configuration = {
    "Batch_Size": 32,
    "Image_Size": 256,
    "Learning_Rate": 0.001,
    "N_Epochs": 5,
    "Dropout_Rate": 0.0,
    "Regularization_Rate": 0.0,
    "N_Filters": 6,
    "Kernel_Size": 3,
    "N_Strides": 1,
    "Pool_Size":2,
    "N_Dense_1": 100,
    "N_Dense_2":10,
    "Num_Class": 2,
    "Class_Name" : ['empty', 'not_empty']
}

In [29]:
Train_Dataset = tf.keras.utils.image_dataset_from_directory(
    Train_Directory,labels='inferred',label_mode ='categorical',
    class_names = Configuration['Class_Name'], color_mode = 'rgb',
    batch_size = Configuration["Batch_Size"], image_size = Configuration["Image_Size"], shuffle = True,
    seed = 99, 
    validation_split=0.2, subset = 'training'
)

Found 6090 files belonging to 2 classes.
Using 4872 files for training.


In [30]:
Resize_Rescaling = tf.keras.Sequential([
    Resizing(Configuration['Image_Size'], Configuration['Image_Size']),
    Rescaling(1.0/255)
])

def Resize_Rescale(image, label):
    return Resize_Rescaling(image, training=True), label

In [31]:
Training_Dataset = (Train_Dataset.map(Resize_Rescale).prefetch(tf.data.AUTOTUNE))

In [38]:
Model = tf.keras.Sequential([
    InputLayer(input_shape=(256,256,3)),
    Resize_Rescaling,

    Conv2D(filters=Configuration["N_Filters"], kernel_size=Configuration["Kernel_Size"], strides=Configuration["N_Strides"], activation='relu', kernel_regularizer=L2(Configuration["Regularization_Rate"])),
    BatchNormalization(),
    MaxPool2D(pool_size=Configuration["Pool_Size"], strides=Configuration["N_Strides"]*2),
    Dropout(rate=Configuration["Dropout_Rate"]),

    Conv2D(filters=Configuration["N_Filters"]*2+4, kernel_size=Configuration["Kernel_Size"], strides=Configuration["N_Strides"], activation='relu', kernel_regularizer=L2(Configuration["Regularization_Rate"])),
    BatchNormalization(),
    MaxPool2D(pool_size=Configuration["Pool_Size"], strides=Configuration["N_Strides"]*2),
    Dropout(rate=Configuration["Dropout_Rate"]),

    Flatten(),

    Dense(Configuration["N_Dense_1"], activation='relu', kernel_regularizer=L2(Configuration["Kernel_Size"])),
    BatchNormalization(),
    Dropout(rate=Configuration["Dropout_Rate"]),

    Dense(Configuration["N_Dense_2"], activation='relu', kernel_regularizer=L2(Configuration["Kernel_Size"])),
    BatchNormalization(),
    
    Dense(Configuration["Num_Class"], activation='softmax')
])

Model.summary()

C:\Users\ABHAY SINHA\AppData\Roaming\Python\Python313\site-packages\keras\src\layers\core\input_layer.py:27: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(


Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ sequential_4 (Sequential)       │ (None, 256, 256, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 254, 254, 6)    │           168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_12          │ (None, 254, 254, 6)    │            24 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 127, 127, 6)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 127, 127, 6)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_7 (Conv2D)               │ (None, 125, 125, 16)   │           880 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_13          │ (None, 125, 125, 16)   │            64 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 62, 62, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 62, 62, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_3 (Flatten)             │ (None, 61504)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 100)            │     6,150,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_14          │ (None, 100)            │           400 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 10)             │         1,010 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_15          │ (None, 10)             │            40 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 2)              │            22 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 6,153,108 (23.47 MB)

 Trainable params: 6,152,844 (23.47 MB)

 Non-trainable params: 264 (1.03 KB)

In [39]:
Loss = CategoricalCrossentropy()

Metrics = [CategoricalAccuracy(name='Accuracy'), TopKCategoricalAccuracy(k=2, name="Top_Accuracy")]

In [40]:
Model.compile(
    optimizer = Adam(learning_rate=Configuration['Learning_Rate']),
    loss = Loss,
    metrics = Metrics
)

In [41]:
History = Model.fit(Training_Dataset, epochs=Configuration["N_Epochs"], verbose=1)

Epoch 1/5
153/153 ━━━━━━━━━━━━━━━━━━━━ 38s 230ms/step - Accuracy: 0.9598 - Top_Accuracy: 1.0000 - loss: 41.3326
Epoch 2/5
153/153 ━━━━━━━━━━━━━━━━━━━━ 35s 227ms/step - Accuracy: 0.9614 - Top_Accuracy: 1.0000 - loss: 3.2053
Epoch 3/5
153/153 ━━━━━━━━━━━━━━━━━━━━ 34s 223ms/step - Accuracy: 0.9661 - Top_Accuracy: 1.0000 - loss: 1.2945
Epoch 4/5
153/153 ━━━━━━━━━━━━━━━━━━━━ 34s 224ms/step - Accuracy: 0.9612 - Top_Accuracy: 1.0000 - loss: 1.3232
Epoch 5/5
153/153 ━━━━━━━━━━━━━━━━━━━━ 35s 228ms/step - Accuracy: 0.9631 - Top_Accuracy: 1.0000 - loss: 1.1557


In [43]:
Model.export('Classifier_Model')
print('Model Saved Successfully')

INFO:tensorflow:Assets written to: Classifier_Model\assets


INFO:tensorflow:Assets written to: Classifier_Model\assets


Saved artifact at 'Classifier_Model'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 256, 256, 3), dtype=tf.float32, name='keras_tensor_41')
Output Type:
  TensorSpec(shape=(None, 2), dtype=tf.float32, name=None)
Captures:
  2548187691536: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2548187692496: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2548187692112: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2548187691920: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2548187691152: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2548187692304: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2548187692688: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2548187691728: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2548190216656: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2548190216464: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2548187692880: TensorSpec(shape

In [44]:
def Represent_Data_Gen():
    for input_value, j in Training_Dataset.take(100):
        yield[input_value]

In [47]:
Converter = tf.lite.TFLiteConverter.from_saved_model('Classifier_Model')
Converter.optimizations = [tf.lite.Optimize.DEFAULT]
Converter.inference_input_type = tf.int8
Converter.inference_output_type = tf.int8
Converter.representative_dataset = Represent_Data_Gen
tflite_model = Converter.convert()

with open('Classifier_Lite_Model', 'wb') as f:
    f.write(tflite_model)

print('Model Saved in tensorflow lite sucessfully')

Model Saved in tensorflow lite sucessfully


In [49]:
import cv2
Test_Image = cv2.imread(r"C:\Users\ABHAY SINHA\Parking_Lot_Project\clf-data\clf-data\not_empty\00000036_00000013.jpg")
Test_Image = cv2.resize(Test_Image, (Configuration['Image_Size'], Configuration['Image_Size']))
Test_Image = np.expand_dims(Test_Image, axis=0)
print(Test_Image.shape)

(1, 256, 256, 3)


In [51]:
Interpreter = tf.lite.Interpreter(model_path='Classifier_Lite_Model')
Interpreter.allocate_tensors()

Input_Details = Interpreter.get_input_details()[0]
Output_Details = Interpreter.get_output_details()[0]

test_image = Test_Image.astype(Input_Details['dtype'])
Interpreter.set_tensor(Input_Details['index'], test_image)

Interpreter.invoke()

Output = Interpreter.get_tensor(Output_Details['index'])[0]

In [52]:
print(np.argmax(Output))
print(Configuration['Class_Name'] [np.argmax(Output)])

1
not_empty
